In [ ]:
# Step 1: Install required packages (if not already installed)
#!pip install langchain langchain-google-genai langgraph requests python-dotenv

In [1]:
# Step 2: Import required libraries
import os
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [4]:
# Load API key from .env file
load_dotenv()

True

In [5]:
# Step 3: Define REST API URLs
PRODUCT_API = "http://127.0.0.1:8000/products"
ORDER_API = "http://127.0.0.1:8002/orders"

In [6]:
# Step 4: Create Tool to fetch products from Product API
@tool
def get_products():
    """Get all product details from the Product API."""
    try:
        response = requests.get(PRODUCT_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Product API: {e}"

In [7]:
# Step 5: Create Tool to fetch orders from Order API
@tool
def get_orders():
    """Get all order details from the Order API."""
    try:
        response = requests.get(ORDER_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Order API: {e}"

In [8]:
# Step 6: Initialize Gemini Model and Tools list
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
        temperature = 0

)

tools = [get_products, get_orders]

In [9]:
# Step 7: Create Agent using LangGraph
agent = create_react_agent(
    model=llm,
    tools=tools
)

C:\Users\SMIT\AppData\Local\Temp\ipykernel_1696\3871573776.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [10]:
# Step 8: Define System Prompt (Strictly answer only products and orders)
system_message = """
You are a product and order chatbot.

You can ONLY answer questions about:
1. Products
2. Orders

For product questions, get the information from the Product API.
For order questions, get the information from the Order API.

Never make up product or order information.

If the user asks anything unrelated to products or orders, say:
"I can only help with product and order information."
"""

In [11]:
# Step 9: Test 1 - Ask about products
question = "What products are available?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "I'm sorry, I am currently having trouble connecting to the Product API. Please try again later.", 'extras': {'signature': 'EmAKXgFpFH0TF37YBmQCZHaqmSklwjHKYEmvoydnnMocg6O4VRdZOIHZFDDnO37ol8sxJw7NNUk3VgyztOqUIpB2350tiMWwHORYtFpNJaTlFP4zV8PBPGpr6NnfA9KLMME='}}]


In [12]:
# Step 10: Test 2 - Ask about specific product details
question = "What is the price of iPhone 15 and is it in stock?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'I apologize, but I am currently having trouble accessing the product information. Please try again later.', 'extras': {'signature': 'EmAKXgFpFH0TYx9k0MVJpwHE0mNjM3qsjCiX1dOG8gnZgRODCPpFfNIpWTFuUFhEUqIAxc5koJhORR7KUDewnCoKhT312HQCXAycuUrYV4cuPpIDMh3RtolmClVGP/YXytA='}}]


In [13]:
# Step 11: Test 3 - Ask about order status
question = "What is the status of order 101?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'I am currently unable to retrieve order information due to a technical issue connecting to the Order API. Please try again later.', 'extras': {'signature': 'EmAKXgFpFH0TESfxTKtVbHQTtChOnG7kTHCWrv4OTS6OTo58rrKar/oTtJX7qmS6o9xXaB7OncD81ZDn2YDMSXtL8oC83HU2zcIEEMo2mHOoMoGD7GtbEZzumXGufosoX0I='}}]


In [14]:
# Step 12: Test 4 - Ask unrelated question (must be refused)
question = "What is the capital of India?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

c:\Users\SMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'I can only help with product and order information.', 'extras': {'signature': 'EmAKXgFpFH0T4xhj4u2ZwNLXeyua7ncjPRE+lNI/aPlJ7eiZjU6CT6O/PNPPtsxzfzKonAPrwCPQPDN73w/Iyx7nB+GQpnB1qQLqMYJCQcQIihft6SBJtVH0jKenpHuHwFI='}}]
